# Part 1: Load, inspect, clean and load into SQL

This notebook follows the project brief's steps in order:

1. **Load** all six CSVs into Pandas DataFrames and **inspect** each one's shape, column data types and null counts.
2. **Clean** each table: fix data types (dates, integer IDs), remove exact duplicate rows, remove orphaned rows.
3. **Leave budget/revenue zeros alone.** TMDB uses 0 for "unknown", so they are *not* silently fixed here.
   How to handle them is decided in the Pandas analysis (`01_eda.ipynb`, section 2).
4. **Load** the cleaned tables into a SQLite database using **SQLAlchemy**.
5. Run a **test JOIN** across `movies`, `movie_genres` and `genres` to confirm the load worked.

The cleaning logic lives in `src/clean.py` and the database load in `src/build_db.py`, so the same
steps can also be run from the command line (`python run_pipeline.py`).

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from config import RAW_DIR, PROCESSED_DIR, DB_PATH, REPORTS_DIR

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

## Step 1: Load the six CSVs and inspect them

In [2]:
FILES = ["movies", "genres", "movie_genres", "cast", "crew", "movie_keywords"]
raw = {name: pd.read_csv(RAW_DIR / f"{name}.csv") for name in FILES}

overview = pd.DataFrame({
    "rows": {n: df.shape[0] for n, df in raw.items()},
    "columns": {n: df.shape[1] for n, df in raw.items()},
    "total nulls": {n: int(df.isna().sum().sum()) for n, df in raw.items()},
    "exact duplicate rows": {n: int(df.duplicated().sum()) for n, df in raw.items()},
})
overview

,rows,columns,total nulls,exact duplicate rows
movies,2509,12,0,6
genres,19,2,0,0
movie_genres,6888,2,0,10
cast,24917,6,11,15
crew,7297,6,0,8
movie_keywords,41701,3,0,20


Next, each table's columns: data type, null count and number of distinct values.

In [3]:
def profile(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nulls": df.isna().sum(),
        "null %": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(),
        "example": df.iloc[0],
    })

for name, df in raw.items():
    print(f"\n=== {name}.csv  shape={df.shape} ===")
    display(profile(df))


=== movies.csv  shape=(2509, 12) ===


,dtype,nulls,null %,unique,example
movie_id,int64,0,0.0,2503,157336
title,str,0,0.0,2452,Interstellar
original_language,str,0,0.0,21,en
release_date,str,0,0.0,2123,2014-11-05
runtime,int64,0,0.0,134,169
budget,float64,0,0.0,347,165000000.0
revenue,float64,0,0.0,2364,746606706.0
popularity,float64,0,0.0,2490,73.0452
vote_average,float64,0,0.0,1554,8.484
vote_count,int64,0,0.0,2132,40566



=== genres.csv  shape=(19, 2) ===


,dtype,nulls,null %,unique,example
genre_id,int64,0,0.0,19,28
genre_name,str,0,0.0,19,Action



=== movie_genres.csv  shape=(6888, 2) ===


,dtype,nulls,null %,unique,example
movie_id,int64,0,0.0,2499,157336
genre_id,int64,0,0.0,19,12


... (2 outputs left out)


=== crew.csv  shape=(7297, 6) ===


,dtype,nulls,null %,unique,example
crew_row_id,int64,0,0.0,7289,1
movie_id,int64,0,0.0,2499,157336
person_id,int64,0,0.0,2848,527
person_name,str,0,0.0,2846,Jonathan Nolan
job,str,0,0.0,3,Writer
department,str,0,0.0,2,Writing



=== movie_keywords.csv  shape=(41701, 3) ===


,dtype,nulls,null %,unique,example
movie_id,int64,0,0.0,2496,157336
keyword_id,int64,0,0.0,10288,1612
keyword_name,str,0,0.0,10288,spacecraft


**What the inspection shows**

- `release_date` is read as **text**, not a date, so it needs converting.
- IDs are integers already, but `budget` and `revenue` are floats (some have fractions of a cent).
- The only nulls are **11 missing `character_name` values** in `cast`.
- Every table except `genres` has **exact duplicate rows**.
- Text columns may have stray whitespace (checked below).

In [4]:
# Problems the profile can't show directly
movies = raw["movies"]
checks = {
    "duplicate movie_id values": int(movies.movie_id.duplicated().sum()),
    "movies with budget = 0": int((movies.budget == 0).sum()),
    "movies with revenue = 0": int((movies.revenue == 0).sum()),
    "keyword names with stray whitespace": int((raw["movie_keywords"].keyword_name
                                                != raw["movie_keywords"].keyword_name.str.strip()).sum()),
}
ids = set(movies.movie_id)
for name in ["movie_genres", "cast", "crew", "movie_keywords"]:
    checks[f"orphaned rows in {name}"] = int((~raw[name].movie_id.isin(ids)).sum())
checks["orphaned genre_id in movie_genres"] = int((~raw["movie_genres"].genre_id.isin(raw["genres"].genre_id)).sum())
pd.Series(checks, name="count").to_frame()

,count
duplicate movie_id values,6
movies with budget = 0,90
movies with revenue = 0,102
keyword names with stray whitespace,34
orphaned rows in movie_genres,0
orphaned rows in cast,0
orphaned rows in crew,0
orphaned rows in movie_keywords,0
orphaned genre_id in movie_genres,0


In [5]:
# The duplicated movie_ids are full copies of the same film, not two different films
movies[movies.movie_id.duplicated(keep=False)].sort_values("movie_id")[
    ["movie_id", "title", "release_date", "budget", "revenue"]]

,movie_id,title,release_date,budget,revenue
56,98,Gladiator,2000-05-04,103000000.0,465516248.0
2504,98,Gladiator,2000-05-04,103000000.0,465516248.0
2501,414,Batman Forever,1995-06-16,100000000.0,336529144.0
902,414,Batman Forever,1995-06-16,100000000.0,336529144.0
2240,8814,Doom,2005-10-20,60000000.0,58100000.0
2502,8814,Doom,2005-10-20,60000000.0,58100000.0
2503,72570,The Vow,2012-02-05,30000000.0,196114570.0
1285,72570,The Vow,2012-02-05,30000000.0,196114570.0
2319,329833,Zoolander 2,2016-02-06,50000000.0,56722693.0
2499,329833,Zoolander 2,2016-02-06,50000000.0,56722693.0


## Step 2: Clean each table

`src/clean.py` applies these rules and logs every one with counts:

| Rule | Tables |
|---|---|
| Parse `release_date` as a date; derive year, month and decade | movies |
| Enforce every `*_id` column as an integer | all |
| Remove exact duplicate rows | all |
| Remove orphaned rows (a `movie_id` not in the cleaned `movies`) | movie_genres, cast, crew, movie_keywords |
| Trim stray whitespace in text | all text columns |
| Keep `budget` / `revenue` values of 0 **unchanged** (only the type changes, to whole dollars) | movies |
| Split repeated names into lookup tables (normalisation) | people, keywords, jobs |

In [6]:
import clean
clean.LOG.clear()
clean.main()

[load] `movies.csv`: 2,509 rows x 12 columns.
[load] `genres.csv`: 19 rows x 2 columns.
[load] `movie_genres.csv`: 6,888 rows x 2 columns.
[load] `cast.csv`: 24,917 rows x 6 columns.
[load] `crew.csv`: 7,297 rows x 6 columns.
[load] `movie_keywords.csv`: 41,701 rows x 3 columns.
[movies] Dropped 6 exact duplicate rows (2509 -> 2503).
[movies] Parsed `release_date` to a date (0 unparseable values).


[movies] Derived `release_year`, `release_month` and `decade`.
[movies] `budget` = 0 on 90 films (TMDB's code for 'unknown'). Kept as 0 in the cleaned data and database, not silently fixed; handled in the Pandas analysis.
[movies] `revenue` = 0 on 102 films (TMDB's code for 'unknown'). Kept as 0 in the cleaned data and database, not silently fixed; handled in the Pandas analysis.
[movies] Flagged (not changed) 6 films with implausibly small revenue under $10,000 (e.g. High School Musical, Return, Society of the Snow); also handled in the Pandas analysis.
[movies] `status` has one value ('Released') for every row - kept for completeness, not analysed.
[movies] IDs confirmed as integers in every table (`movie_id`, `genre_id`, `person_id`, `keyword_id`, row ids).
[movies] Range checks passed: runtime 45-242 min, vote_average 0-10, vote_count >= 0.
[movie_genres] Dropped 10 exact duplicate rows: 6888 -> 6878.
[movie_genres] Referential check: every movie_id exists in movies.


[movie_keywords] Trimmed stray whitespace in `keyword_name` on 34 rows.


[movie_keywords] Dropped 20 exact duplicate rows: 41701 -> 41681.
[movie_keywords] Referential check: every movie_id exists in movies.


[keywords] Extracted 10,288 distinct keywords into a lookup table (3NF).


[cast] Dropped 15 exact duplicate rows: 24917 -> 24902.
[cast] Referential check: every movie_id exists in movies.
[cast] 11 credits have no character name - kept as NULL (the credit itself is valid).
[crew] Dropped 8 exact duplicate rows: 7297 -> 7289.
[crew] Referential check: every movie_id exists in movies.
[people] Built `people` from cast + crew: 13,960 unique people (188 appear in both).
[jobs] Moved department into a `jobs` lookup (3 jobs) since department depends on job.
[coverage] 4 films have no genre rows - kept, they simply drop out of genre analyses.
[coverage] 5 films have no cast rows - kept, they simply drop out of cast analyses.
[coverage] 7 films have no keyword rows - kept, they simply drop out of keyword analyses.
[coverage] 4 films have no director credited.
[output] `data/processed/movies.csv`: 2,503 rows.
[output] `data/processed/genres.csv`: 19 rows.
[output] `data/processed/movie_genres.csv`: 6,878 rows.
[output] `data/processed/keywords.csv`: 10,288 rows.
[ou

[output] `data/processed/people.csv`: 13,960 rows.


[output] `data/processed/cast_credits.csv`: 24,902 rows.
[output] `data/processed/jobs.csv`: 3 rows.


[output] `data/processed/crew_credits.csv`: 7,289 rows.


In [7]:
cleaned_movies = pd.read_csv(PROCESSED_DIR / "movies.csv", parse_dates=["release_date"])
print("Data types after cleaning:")
cleaned_movies[["movie_id", "release_date", "release_year", "budget", "revenue", "vote_count"]].dtypes.to_frame("dtype")

Data types after cleaning:


,dtype
movie_id,int64
release_date,datetime64[us]
release_year,int64
budget,int64
revenue,int64
vote_count,int64


In [8]:
# Budget/revenue zeros are still there - on purpose
pd.Series({
    "budget = 0 (raw)": int((raw["movies"].budget == 0).sum()),
    "budget = 0 (cleaned)": int((cleaned_movies.budget == 0).sum()),
    "revenue = 0 (raw)": int((raw["movies"].revenue == 0).sum()),
    "revenue = 0 (cleaned)": int((cleaned_movies.revenue == 0).sum()),
}, name="films").to_frame()

,films
budget = 0 (raw),90
budget = 0 (cleaned),90
revenue = 0 (raw),102
revenue = 0 (cleaned),102


In [9]:
# Before vs after: rows per table, and how each original CSV maps to the database
mapping = pd.DataFrame([
    ("movies.csv", "movies", len(raw["movies"]), "movies"),
    ("genres.csv", "genres", len(raw["genres"]), "genres"),
    ("movie_genres.csv", "movie_genres", len(raw["movie_genres"]), "movie_genres"),
    ("cast.csv", "cast_credits", len(raw["cast"]), "cast_credits (+ names in people)"),
    ("crew.csv", "crew_credits", len(raw["crew"]), "crew_credits (+ names in people, job in jobs)"),
    ("movie_keywords.csv", "movie_keywords", len(raw["movie_keywords"]), "movie_keywords (+ names in keywords)"),
], columns=["original CSV", "processed", "rows before", "database table(s)"])
mapping["rows after"] = [len(pd.read_csv(PROCESSED_DIR / f"{t}.csv")) for t in mapping["processed"]]
mapping["removed"] = mapping["rows before"] - mapping["rows after"]
mapping[["original CSV", "rows before", "rows after", "removed", "database table(s)"]]

,original CSV,rows before,rows after,removed,database table(s)
0,movies.csv,2509,2503,6,movies
1,genres.csv,19,19,0,genres
2,movie_genres.csv,6888,6878,10,movie_genres
3,cast.csv,24917,24902,15,cast_credits (+ names in people)
4,crew.csv,7297,7289,8,"crew_credits (+ names in people, job in jobs)"
5,movie_keywords.csv,41701,41681,20,movie_keywords (+ names in keywords)


The full rule-by-rule log is saved to `reports/data_cleaning_log.md`.

**Why nine tables from six files?** In the raw files the same person name is repeated on every credit,
the same keyword name on every movie-keyword row, and the department next to every job. Moving these
into `people`, `keywords` and `jobs` lookups (third normal form) means every fact is stored once.
All six original tables are still there; three small lookup tables are added.

## Step 3: Load the cleaned tables into SQLite with SQLAlchemy

In [10]:
from sqlalchemy import create_engine, event, inspect, text
from build_db import LOAD_ORDER, create_schema

if DB_PATH.exists():
    DB_PATH.unlink()
engine = create_engine(f"sqlite:///{DB_PATH}")

@event.listens_for(engine, "connect")
def enforce_foreign_keys(dbapi_con, _):
    dbapi_con.execute("PRAGMA foreign_keys = ON")

# 1) Create tables, primary keys, foreign keys, constraints and indexes from sql/schema.sql
create_schema(engine)

# 2) Load each cleaned table with DataFrame.to_sql - parent tables first so foreign keys resolve
loaded = {}
for table in LOAD_ORDER:
    df = pd.read_csv(PROCESSED_DIR / f"{table}.csv")
    if table == "movies":
        df["release_date"] = pd.to_datetime(df["release_date"]).dt.strftime("%Y-%m-%d")
    df.to_sql(table, engine, if_exists="append", index=False, chunksize=5000)
    with engine.connect() as con:
        loaded[table] = con.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar_one()

pd.Series(loaded, name="rows in database").to_frame()

,rows in database
movies,2503
genres,19
keywords,10288
people,13960
jobs,3
movie_genres,6878
movie_keywords,41681
cast_credits,24902
crew_credits,7289


In [11]:
# 3) Check the structure and the foreign keys
insp = inspect(engine)
print("Tables:", insp.get_table_names())
print("Views: ", insp.get_view_names())
with engine.connect() as con:
    fk_problems = con.execute(text("PRAGMA foreign_key_check")).fetchall()
print("Foreign key violations:", len(fk_problems))
pd.DataFrame(insp.get_foreign_keys("cast_credits"))[["constrained_columns", "referred_table", "referred_columns"]]

Tables: ['cast_credits', 'crew_credits', 'genres', 'jobs', 'keywords', 'movie_genres', 'movie_keywords', 'movies', 'people']
Views:  ['v_movie_summary']
Foreign key violations: 0


,constrained_columns,referred_table,referred_columns
0,[person_id],people,[person_id]
1,[movie_id],movies,[movie_id]


## Step 4: Test query - JOIN movies, movie_genres and genres

In [12]:
test_sql = text("""
    SELECT m.title, m.release_year, g.genre_name
    FROM movies m
    JOIN movie_genres mg ON mg.movie_id = m.movie_id
    JOIN genres g        ON g.genre_id  = mg.genre_id
    WHERE m.title IN ('Interstellar', 'Inception', 'The Dark Knight')
    ORDER BY m.title, g.genre_name
""")
pd.read_sql(test_sql, engine)

,title,release_year,genre_name
0,Inception,2010,Action
1,Inception,2010,Adventure
2,Inception,2010,Science Fiction
3,Interstellar,2014,Adventure
4,Interstellar,2014,Drama
5,Interstellar,2014,Science Fiction
6,The Dark Knight,2008,Action
7,The Dark Knight,2008,Crime
8,The Dark Knight,2008,Thriller


In [13]:
# The same JOIN aggregated over the whole database
pd.read_sql(text("""
    SELECT g.genre_name, COUNT(*) AS films
    FROM movies m
    JOIN movie_genres mg ON mg.movie_id = m.movie_id
    JOIN genres g        ON g.genre_id  = mg.genre_id
    GROUP BY g.genre_name
    ORDER BY films DESC
"""), engine)

,genre_name,films
0,Drama,937
1,Action,808
2,Comedy,762
3,Adventure,674
4,Thriller,673
5,Science Fiction,463
6,Fantasy,396
7,Crime,387
8,Romance,349
9,Family,344


The JOIN returns the expected genres for well-known films (Inception: Action, Adventure,
Science Fiction), every genre has films attached, and the foreign-key check found no violations.
The database loaded correctly, so we can move on to the SQL analysis (`sql/analysis_queries.sql`)
and the Pandas EDA (`01_eda.ipynb`).

In [14]:
engine.dispose()